# 06 · EDA Finanzas e inversión pública

Una fuente en `data/raw/FINANZAS_INVERSION_PUBLICA/`:
- `inversion_educacion_por_localidad_12_2025.gpkg` — territorialización de la inversión en
  educación por localidad (capa `sed_2026__territorializacioninversion2025`, 20 polígonos,
  EPSG:3857), con asignado, ejecutado y girado.

**Indicadores objetivo**: FIN-01 (ejecución presupuestal por localidad), FIN-02 (girado vs
asignado). Las unidades parecen ser pesos COP; validar con la SED.

## 0. Configuración

In [ ]:
import sys, os, re, json, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
warnings.filterwarnings("ignore")

if (Path("../..") / "src" / "eda").exists():
    ROOT = Path("../..").resolve()
elif (Path("..") / "src" / "eda").exists():
    ROOT = Path("..").resolve()
elif (Path(".") / "src" / "eda").exists():
    ROOT = Path(".").resolve()
else:
    ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw"
MR_PATH = RAW_DIR / "INFRAESTRUCTURA_ESPACIO_PUBLICO" / "gpkg_mr_v03.26" / "gpkg_mr_v03.26.gpkg"
REPORTS = ROOT / "reports" / "eda"
PERFILES = REPORTS / "perfiles"
CACHE = REPORTS / "cache"
TIEMPOS = REPORTS / "tiempos"
for _d in (REPORTS, PERFILES, CACHE, TIEMPOS):
    _d.mkdir(parents=True, exist_ok=True)

SMOKE = os.environ.get("EDA_SMOKE") == "1"
print("ROOT:", ROOT, "| SMOKE:", SMOKE)

import src.eda as eda
from src.eda.explore import explorar_dataset

_SECTION_T = {}

def t0(nombre):
    _SECTION_T[nombre] = time.time()

def t1(nombre):
    _SECTION_T[nombre] = time.time() - _SECTION_T[nombre]

def guardar_tiempos(archivo):
    df = pd.DataFrame([{"seccion": k, "segundos": round(v, 1)} for k, v in _SECTION_T.items()])
    df.to_csv(TIEMPOS / archivo, index=False)
    print(f"tiempos guardados: {archivo} ({len(df)} secciones)")

## 0.1 Fuentes del sector en el catálogo

In [ ]:
cat = eda.load_catalog()
sec = cat[cat["indicadores"].astype(str).str.contains("FIN", case=False, na=False)]
display(sec[["id", "nombre", "archivo", "temporalidad", "indicadores", "valor_publico"]])

## 1. Inversión en educación por localidad

### Territorialización de inversión en educación (2025)

In [ ]:
t0('inversion')
SPEC = {
    'id': 'inversion_educacion_2025',
    'titulo': 'Territorialización de inversión en educación (2025)',
    'path': 'data/raw/FINANZAS_INVERSION_PUBLICA/inversion_educacion_por_localidad_12_2025.gpkg',
    'capa': 'sed_2026__territorializacioninversion2025',
    'origen': 'Secretaría de Educación del Distrito (SED)',
    'corte': 'Dic 2025 (nombre del archivo)',
    'valor_publico': 'Asignación y ejecución de inversión en educación por localidad',
    'indicadores': 'FIN-01 (ejecución presupuestal), FIN-02 (girado)',
    'notas': '20 polígonos (MultiPolygon, EPSG:3857). Columnas: R_ASIGNADOS, R_EJECUTADOS, R_GIRADOS, COD_LOCA. COD_LOCA numérico 1-20; unidades por validar (pesos COP).',
}
RES = explorar_dataset(SPEC, RAW_DIR, PERFILES, smoke=SMOKE)
t1('inversion')

## 2. Análisis de ejecución presupuestal

### 2.1 Asignado, ejecutado y girado por localidad

In [ ]:
from src.eda.profiling import localidad_de_codigo
fin = gpd.read_file(str(RAW_DIR / "FINANZAS_INVERSION_PUBLICA" / "inversion_educacion_por_localidad_12_2025.gpkg"), layer="sed_2026__territorializacioninversion2025")
fin["LOCALIDAD"] = fin["COD_LOCA"].map(localidad_de_codigo)
fin["PCT_EJEC"] = fin["R_EJECUTADOS"] / fin["R_ASIGNADOS"]
fin["PCT_GIRADO"] = fin["R_GIRADOS"] / fin["R_ASIGNADOS"]
tabla = fin[["LOCALIDAD", "R_ASIGNADOS", "R_EJECUTADOS", "R_GIRADOS", "PCT_EJEC", "PCT_GIRADO"]].sort_values("R_ASIGNADOS", ascending=False)
tabla.to_csv(REPORTS / "finanzas_inversion_por_localidad.csv", index=False)
display(tabla)
print(f"Total asignado: {fin['R_ASIGNADOS'].sum():,.0f} | ejecutado: {fin['R_EJECUTADOS'].sum():,.0f} ({fin['R_EJECUTADOS'].sum() / fin['R_ASIGNADOS'].sum():.1%})")
print(f"Ejecución promedio por localidad: {fin['PCT_EJEC'].mean():.1%} (min {fin['PCT_EJEC'].min():.1%}, max {fin['PCT_EJEC'].max():.1%})")
tabla.plot(x="LOCALIDAD", y="PCT_EJEC", kind="bar", figsize=(11, 4), title="% de ejecución por localidad", color="#4c72b0")
plt.ylabel("% ejecutado")
plt.show()

### 2.2 Mapa temático de ejecución

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
fin.plot(ax=ax, column="PCT_EJEC", cmap="RdYlGn", legend=True, edgecolor="white", linewidth=0.4, legend_kwds={"label": "% ejecutado", "shrink": 0.6})
ax.set_title("Ejecución de inversión en educación por localidad (2025)")
ax.set_axis_off()
plt.show()

## 3. Indicadores del sector

In [ ]:
sts = eda.indicator_status()
sts_sec = sts[sts["indicador"].str.startswith("FIN", na=False)]
display(sts_sec[["indicador", "dimension", "estado", "que_falta"]])
sts_sec.to_csv(REPORTS / "indicadores_finanzas.csv", index=False)
guardar_tiempos("06_eda_finanzas.csv")
print("Secciones del notebook:", list(_SECTION_T.keys()))